# unit01 レッスン: コンペの解剖と最初の提出

**題材** — 中古品マーケットプレイスの出品データから、**出品価格 `price` を当てる**回帰コンペ。
評価指標は **RMSLE**(あとで説明する)。データは `data/` に同梱済みで、ネットワークは一切使わない。

## このレッスンを終えると作れるようになるもの

1. コンペのデータ一式(`train.csv` / `test.csv` / `sample_submission.csv`)を受け取って、
   「何を予測するのか」「どんな形で提出するのか」「どう採点されるのか」を自分の言葉で説明できる
2. pandas で CSV を読み、**行数・列・型・分布**を確認し、条件で行を絞り込める
3. 「モデルを作る前に入力を疑う」健全性チェックを走らせ、**このデータに仕込まれた壊れ方を全部見つけられる**
4. フォーマット検証つきの `submission.csv` を2種類(定数 → カテゴリ別中央値)作り、スコアを比較できる

所要の目安: **90〜120分**。このあと演習 `ex01`〜`ex04` が続く。

## このレッスンの読み方

セルは**上から順に**実行する。構成は概念ごとに次の8ステップの繰り返し:

| 記号 | 内容 |
|---|---|
| ① | なぜこれを学ぶのか(実務のどこで使うか) |
| ② | 解説(C# との対応表・API 表) |
| ③ | **見る** — 完成コードを実行して結果を見る |
| ④ | **予測する** — 次のセルの結果を頭の中で予測する |
| ⑤ | **変えてみる** — 実行して予測と照合する |
| ⑥ | **書いてみる**(指示) |
| ⑦ | 自分で書くセル(`# ここに書く`) |
| ⑧ | チェックポイント(即時採点) |

⑦ を書かずに実行しても notebook は止まらない。⑧ が `[NG]` を出して、何が期待値なのかを教えてくれる。

In [ ]:
# ===== セットアップ: このセルを最初に1回だけ実行する =====
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)

# データの場所。notebook をユニット直下で開いても、リポジトリのルートで開いても動くようにする。
DATA = Path("data")
if not (DATA / "train.csv").exists():
    DATA = Path("courses/kaggle-sprint/unit01-competition-anatomy/data")
assert (DATA / "train.csv").exists(), f"train.csv が見つかりません: {DATA.resolve()}"

# 作った submission.csv の置き場所
OUT = DATA.parent / "output"
OUT.mkdir(exist_ok=True)

print("pandas:", pd.__version__, "/ numpy:", np.__version__)
print("DATA =", DATA.resolve())
print("OUT  =", OUT.resolve())


# ---------- 採点ヘルパー(中身は読まなくてよい) ----------
def check(name, actual, expected, hint=""):
    import numpy as _np
    try:
        ok = actual is not None and bool(_np.all(_np.isclose(_np.asarray(actual, dtype=float), _np.asarray(expected, dtype=float))))
    except (TypeError, ValueError):
        ok = actual == expected
    if ok:
        print(f"[OK] {name}: 正解!")
    else:
        print(f"[NG] {name}: 期待値 {expected!r} / 実際 {actual!r}")
        if hint:
            print(f"     ヒント: {hint}")
    return ok


def check_frame(name, actual, shape=None, columns=None, hint=""):
    """DataFrame の形と列名を採点する。DataFrame でなくても例外にしない。"""
    if not isinstance(actual, pd.DataFrame):
        print(f"[NG] {name}: 期待値 pandas.DataFrame(shape={shape}, columns={columns}) / 実際 {type(actual).__name__}")
        if hint:
            print(f"     ヒント: {hint}")
        return False
    problems = []
    if shape is not None and tuple(actual.shape) != tuple(shape):
        problems.append(f"shape の期待値 {tuple(shape)} / 実際 {tuple(actual.shape)}")
    if columns is not None and list(actual.columns) != list(columns):
        problems.append(f"列名の期待値 {list(columns)} / 実際 {list(actual.columns)}")
    if problems:
        print(f"[NG] {name}: " + " | ".join(problems))
        if hint:
            print(f"     ヒント: {hint}")
        return False
    print(f"[OK] {name}: 正解!")
    return True


def call_safely(fn, *args, **kwargs):
    """未完成の関数を呼んでも notebook が止まらないようにするラッパ。例外なら None を返す。"""
    if not callable(fn):
        return None
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print(f"     (関数の中で例外が出ました → {type(e).__name__}: {e})")
        return None


def frame_stat(df, col, how):
    """df[col] の統計を安全に取り出す。取れなければ None を返す(未記入でも止まらないため)。"""
    if not isinstance(df, pd.DataFrame) or col not in df.columns:
        return None
    s = df[col]
    table = {"min": s.min, "max": s.max, "mean": s.mean, "sum": s.sum,
             "nunique": s.nunique, "isna_sum": lambda: s.isna().sum()}
    try:
        return table[how]()
    except Exception:
        return None


print("\nセットアップ完了。ヘルパー: check / check_frame / call_safely / frame_stat")

---
# 概念1 — コンペの解剖

## ① なぜ: 最初の30分で勝負の半分が決まる

新しいコンペを開いた日、あるいは実務で「このデータで売上を予測して」と CSV 一式を渡された日、
**最初の30分にやるべきことは決まっている**。データを眺めることではなく、**課題の構造を確定させること**だ。

- 予測する対象は何の列か(= 目的変数)
- 何を手掛かりに使ってよいのか(= 特徴量)
- 提出物はどんな形か(列名・行数・順序)
- **どう採点されるのか**(= 評価指標)

ここを取り違えると、そのあと何時間モデルを磨いても全部無駄になる。
実務でも同じで、「精度99%出ました」と持っていったら**評価軸が違っていた**、は本当によくある事故だ。
だからモデルの話より先に、この4つを固める。

## ② 解説: コンペは3つのファイルと1つの数式でできている

### 3つのファイル

| ファイル | 目的変数 `price` | 何者か | C# で言うと |
|---|---|---|---|
| `train.csv` | **ある** | 答え付きの問題集。ここから規則を学ぶ | 期待値つきのテストケース集 |
| `test.csv` | **ない** | 答えを隠した問題。ここに対する予測を提出する | 実行結果しか見えない隠しテスト |
| `sample_submission.csv` | ある(ダミー値) | **提出フォーマットの唯一の正解** | インターフェース定義。中身は自由、形は固定 |

**`test.csv` に目的変数が無い**のがコンペの本質だ。だから「test を見ながらチューニングする」ことは原理的にできず、
手元で自分の実力を測る仕組み(= 検証設計。unit02 でやる)が必要になる。

**`sample_submission.csv` が提出フォーマットの唯一の正解**というのも重要。
コンペページの説明文と実ファイルが食い違っていたら、**実ファイルが正**。
だから提出ファイルは「ゼロから組み立てる」のではなく「`sample_submission` を読んで `price` 列を差し替える」のが安全な作り方になる。

### 評価指標: この課題は RMSLE

**RMSLE** = Root Mean Squared **Logarithmic** Error(対数を取ってからの二乗平均平方根誤差)。

$$\mathrm{RMSLE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}\bigl(\log(1+\hat{y}_i) - \log(1+y_i)\bigr)^2}$$

普通の RMSE(対数なし)との違いは一点だけ、**誤差を「差」で測るか「比」で測るか**。

- RMSE は「1000円ズレた」を絶対額で測る。→ **高額商品の誤差だけで指標が決まってしまう**
- RMSLE は対数空間で測るので「**何倍ズレたか**」を見る。500円の商品を1000円と予測(2倍)も、
  1万円の商品を2万円と予測(2倍)も、同じ重さのペナルティになる

この課題の `price` は**対数正規分布**(安い商品が大量にあり、たまに数十万円の商品がある = 裾が重い)なので、
RMSE を使うと「たまにある高額商品を当てるゲーム」に化けてしまう。RMSLE ならどの価格帯も公平に効く。

`log(1+y)` の **+1** は「価格0円のとき $\log 0 = -\infty$ になるのを避ける」ため。NumPy では `np.log1p(y)` がこれ。

> **指標が決まると戦略が決まる。** RMSLE なら「対数空間で予測すればよい」、つまり
> 平均値ではなく**中央値**(対数空間での中心に近い)を出す方が有利になる。今日の最初の提出でこれを実感する。

### public LB と private LB

提出すると出るスコアは、実は test の**一部**でしか計算されていない。

| | 計算対象 | いつ見える | 役割 |
|---|---|---|---|
| **public LB** | test の一部(例: 30%) | 提出直後 | 進捗の目安 |
| **private LB** | 残り(例: 70%) | コンペ終了後 | **最終順位はこっち** |

public LB を見ながらチューニングすると、public の30%に**過剰適合**して、終了後に順位が大崩れする(= shake down)。
だから頼るべきは public LB ではなく**手元の検証スコア**になる。このレッスンの最後で、
君の提出の public / private の両方を実際に見せる。

In [ ]:
# GOAL: コンペのデータ一式が「何と何でできているか」を、shape と列名の差分だけで掴む

# STEP 1: 3つのファイルを読み込む。読み込んだら必ず shape(= 行数と列数のタプル)を print する。
#          これはこのコース全体を通しての規律。行数がどこで変わったかを常に追えるようにする。
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
sample_submission = pd.read_csv(DATA / "sample_submission.csv")

print("train            :", train.shape)   # (行数, 列数)
print("test             :", test.shape)
print("sample_submission:", sample_submission.shape)

# STEP 2: train と test の「列の差分」= 予測すべきもの。
#          set は Python の集合型(C# の HashSet<T>)。- で差集合が取れる。
only_in_train = sorted(set(train.columns) - set(test.columns))
only_in_test = sorted(set(test.columns) - set(train.columns))
print("\ntrain にしかない列(= 目的変数):", only_in_train)
print("test にしかない列              :", only_in_test)

# STEP 3: sample_submission の中身 = 提出フォーマットの正解
print("\n--- sample_submission の先頭3行 ---")
print(sample_submission.head(3))
print("列名:", list(sample_submission.columns))
print("price のユニーク値の個数:", sample_submission["price"].nunique(), "→ 全行が同じ定数(ただのひな型)")

# STEP 4: 目的変数の分布をざっと見る。裾が重いかどうかは「平均 vs 中央値」の差で分かる。
print("\n--- price の分布 ---")
print("件数  :", len(train))
print("平均  :", round(train["price"].mean(), 1))
print("中央値:", train["price"].median())
print("最大  :", train["price"].max())
print("最小  :", train["price"].min())
print("\n平均が中央値の2倍以上 → 少数の巨大値に引っ張られている = 裾が重い分布のサイン")

## ④ 予測: RMSE と RMSLE で「ひどさ」の判定はどう変わる?

次のセルでは、正解が **`[50円, 10000円]` の2件**に対して、2通りの予測を採点する。

| | 50円の商品への予測 | 10000円の商品への予測 | どういう外し方か |
|---|---|---|---|
| 予測A | **100円** | 10000円 | 安い方を **2倍** に外した |
| 予測B | 50円 | **20000円** | 高い方を **2倍** に外した |

実行する前に、紙かコメントに答えを書いてみよう。

1. **RMSE**(対数なし)で採点したら、A と B のどちらが「ひどい予測」と判定される? だいたい何倍の差がつく?
2. **RMSLE** で採点したら? A と B の差は大きい? 小さい?
3. この違いは、**中古品の価格予測というタスク**にとってどちらが望ましい?

In [ ]:
# GOAL: 同じ「2倍外し」でも、指標を変えると評価が真逆になることを数値で見る

y_true = np.array([50.0, 10000.0])
pred_a = np.array([100.0, 10000.0])   # 安い方を2倍に外した
pred_b = np.array([50.0, 20000.0])    # 高い方を2倍に外した


# STEP 1: RMSE — 差をそのまま二乗して平均、その平方根
def rmse_demo(y_t, y_p):
    return float(np.sqrt(np.mean((y_p - y_t) ** 2)))


# STEP 2: RMSLE — np.log1p(x) = log(1 + x) に移してから同じことをする
#          np.log1p は「1 を足してから自然対数」を精度よく計算する NumPy 関数。
def rmsle_demo(y_t, y_p):
    return float(np.sqrt(np.mean((np.log1p(y_p) - np.log1p(y_t)) ** 2)))


print(f"{'':<26}{'RMSE':>12}{'RMSLE':>12}")
print(f"{'予測A(安い方を2倍)':<22}{rmse_demo(y_true, pred_a):>12.4f}{rmsle_demo(y_true, pred_a):>12.4f}")
print(f"{'予測B(高い方を2倍)':<22}{rmse_demo(y_true, pred_b):>12.4f}{rmsle_demo(y_true, pred_b):>12.4f}")

print("\nRMSE では B が A の約", round(rmse_demo(y_true, pred_b) / rmse_demo(y_true, pred_a)), "倍ひどい判定")
print("RMSLE では B / A =", round(rmsle_demo(y_true, pred_b) / rmsle_demo(y_true, pred_a), 3), "→ ほぼ同じ判定")

# STEP 3: 定数で当てにいくとき、平均と中央値のどちらが RMSLE に有利かを実データで確認する
const_mean = np.full(len(train), train["price"].mean())
const_median = np.full(len(train), train["price"].median())
print("\n--- train 全体を1つの定数で予測したときの RMSLE ---")
print("平均値を出す  :", round(rmsle_demo(train["price"].to_numpy(dtype=float), const_mean), 5))
print("中央値を出す  :", round(rmsle_demo(train["price"].to_numpy(dtype=float), const_median), 5))
print("→ 裾の重い分布 + RMSLE では、平均より中央値の方が強い")

## ⑥ 書いてみる: RMSLE を自分で実装する

指標は「ライブラリから呼ぶもの」である前に「**自分で書けるもの**」であるべきだ。
式が手に入っていれば、スコアが動いた理由を式に戻って考えられる。

次のセルの `rmsle` 関数を完成させよう。

- 引数 `y_true`, `y_pred` は同じ長さの1次元配列(NumPy 配列でも Python のリストでも動くようにする)
- 返り値は **Python の float** ひとつ
- 使う道具: `np.asarray(x, dtype=float)`(配列に変換)、`np.log1p`、`np.mean`、`np.sqrt`

$$\mathrm{RMSLE} = \sqrt{\frac{1}{n}\sum_i \bigl(\log(1+\hat{y}_i) - \log(1+y_i)\bigr)^2}$$

3行あれば書ける。既習の NumPy の**要素ごとの演算**がそのまま使える(ループは不要)。

In [ ]:
def rmsle(y_true, y_pred):
    """RMSLE を返す。y_true / y_pred は同じ長さの1次元配列。返り値は float。"""
    # ここに書く(ヒント: np.log1p で対数に移してから、差の2乗 → 平均 → 平方根)
    return None

In [ ]:
# ===== チェックポイント 1: RMSLE =====
_yt = np.array([50.0, 10000.0])

check("A-1 完全一致なら 0 になる",
      call_safely(rmsle, _yt, _yt), 0.0,
      hint="同じ配列どうしなら差が全部0。log1p を掛けても0のまま。")

check("A-2 安い方を2倍に外したとき",
      call_safely(rmsle, _yt, np.array([100.0, 10000.0])), 0.4831624461091602,
      hint="log1p(100) - log1p(50) を2乗し、要素数2で平均してから sqrt する。差の順序(pred - true)は2乗するのでどちらでもよい。")

check("A-3 3件のケース",
      call_safely(rmsle, [100.0, 1000.0, 10000.0], [120.0, 900.0, 11000.0]), 0.1326667740619757,
      hint="Python のリストを渡しても動くように np.asarray(x, dtype=float) で変換してから計算する。")

check("A-4 返り値は Python の float か",
      1.0 if isinstance(call_safely(rmsle, _yt, _yt), float) else 0.0, 1.0,
      hint="np.sqrt(...) は NumPy のスカラーを返す。float(...) で包んで返す。")

print("\n(4つとも [OK] になったら次の概念へ)")

---
# 概念2 — pandas を実戦投入する

## ① なぜ: 表データの仕事の8割は「読む・見る・絞る」

コンペでも実務でも、モデルを書いている時間より、**表を読み込んで・形を確かめて・条件で切り出している**時間の方が長い。
「本・音楽カテゴリだけ抜き出して価格の分布を見たい」「送料込みの出品だけで平均を出したい」
「先月以降のデータに絞りたい」— こういう操作を1行で書けるかどうかが、そのまま調査の速度になる。

C# なら `List<T>` に LINQ を生やして `Where` / `Select` / `GroupBy` する場面だ。
Python でそれをやるのが **pandas** で、しかも pandas は**列方向にも切れる**(C# の匿名型の射影より強い)。
ここでは「コンペを解くために今すぐ必要な API」だけを、必要な順に入れる。

## ② 解説: DataFrame は「LINQ の効く表」

### C# との対応表

| やりたいこと | C#(LINQ) | pandas | 返るもの |
|---|---|---|---|
| 表そのもの | `List<匿名型>` | `DataFrame` | — |
| 1列ぶん | `T[]` + 添字 | `Series` | 値の並び + **インデックス**(行ラベル) |
| 行数 | `list.Count` | `len(df)` / `df.shape[0]` | int |
| 条件で行を絞る | `list.Where(x => x.Price > 0)` | `df[df["price"] > 0]` | DataFrame |
| 列を1本取る | `list.Select(x => x.Price)` | `df["price"]` | **Series** |
| 列を複数取る | `list.Select(x => new { x.Category, x.Price })` | `df[["category", "price"]]` | **DataFrame** |
| 行と列を同時に | (直接の対応なし) | `df.loc[行条件, 列リスト]` | DataFrame |
| 重複除去 | `list.Distinct()` | `df.drop_duplicates()` | DataFrame |

**`df["price"]`(角括弧1つ)は Series、`df[["price"]]`(角括弧2つ)は1列だけの DataFrame**。
これは初学者が必ず一度は踏む段差なので、③ のセルで型と shape を print して確かめる。

`Series` は「1列ぶんの `T[]` に**行ラベル**が付いたもの」。C# の `T[]` と違うのは、
**足し算や比較のときに行ラベルで自動的に突き合わせが起きる**こと。今日はまだ使わないが、頭の隅に置いておく。

### 既習の NumPy がそのまま効く

君は `numpy-masking` を習得済みなので、実は新しく覚えることはほとんどない。

```python
arr[arr > 0]                    # NumPy: ブールマスクで抽出
df[df["price"] > 0]             # pandas: まったく同じ発想
```

`df["price"] > 0` は**ブール値の Series**(NumPy のブール配列に行ラベルが付いたもの)を返し、
それを `df[...]` に渡すと True の行だけが残る。**注意点も NumPy と同じ**:

- 複数条件は `and` / `or` ではなく **`&` / `|` / `~`**
- 演算子の優先順位の都合で、**各条件を括弧で囲む**: `df[(df["a"] > 0) & (df["b"] == 1)]`

### 今日使う API 一覧

| 用途 | API | 戻り値 | 注意 |
|---|---|---|---|
| CSV を読む | `pd.read_csv(path, dtype=..., parse_dates=[...])` | DataFrame | `parse_dates` を指定しないと日時列が**ただの文字列**のまま |
| 形を見る | `df.shape` | `(行数, 列数)` のタプル | **属性なので括弧を付けない** |
| 型を見る | `df.dtypes` | Series | 整数のはずの列が `float64` → **欠損を疑う** |
| 先頭を見る | `df.head(n)` | DataFrame | 既定は5行 |
| 概要を見る | `df.info()` | `None`(印字するだけ) | 列ごとの non-null 件数とメモリが見える |
| 数値を要約 | `df.describe()` | DataFrame | 件数・平均・標準偏差・四分位 |
| 列を選ぶ | `df["col"]` / `df[["a","b"]]` | Series / DataFrame | 角括弧の数で型が変わる |
| 行を絞る | `df[bool_series]` | DataFrame | `Where` 相当 |
| 行+列 | `df.loc[行条件, 列]` | DataFrame / Series | **代入するときはこちらを使う**(下記) |

### pandas 3.0 系の注意 — ネットの Kaggle ノートブックは 1.x / 2.x 前提が多い

この環境の pandas は **3.0 系**。検索して出てくるコードとは以下が食い違う。先に潰しておく。

| 変わった点 | 1.x / 2.x のネット記事 | pandas 3.0(この環境) | 実害と対処 |
|---|---|---|---|
| **Copy-on-Write が既定** | `df[cond]["col"] = x` で元が書き換わることがあった | **書き換わらない**。`ChainedAssignmentError` 警告が出るだけ | 「代入したのに反映されない」。→ **`df.loc[cond, "col"] = x`** と1回の `.loc` で書く |
| **文字列列の既定 dtype** | `object` | **`str`** | `select_dtypes(include="object")` が空振りする。→ `include="str"` |
| `df.append(...)` | あった | **削除済み** | → `pd.concat([df1, df2], ignore_index=True)` |
| `inplace=True` | 多用されていた | 非推奨方向 | → 戻り値を受け取る書き方(`df = df.drop_duplicates()`)に統一 |

**Copy-on-Write** とは「見た目はコピーだが、書き換えるまで実体を共有する」方式のこと。
`df[cond]` が返すのは**新しい別オブジェクト**なので、そこに代入しても元の `df` には届かない。
C# で `list.Where(...)` の結果に代入しても元のリストが変わらないのと同じ、と思えば自然な挙動だ。

In [ ]:
# GOAL: 読み込み時に型を指定し、「形・型・中身・分布」を5つのAPIで一気に掴む

# STEP 1: dtype と parse_dates を指定して読み直す
#   dtype={"shipping": "int8"} — 0/1 しか入らない列を1バイト整数にする(メモリ節約。大きいコンペで効く)
#   parse_dates=["listed_at"] — 文字列ではなく日時型として読む。指定しないと ".dt.month" などが使えない
train = pd.read_csv(DATA / "train.csv", dtype={"shipping": "int8"}, parse_dates=["listed_at"])
test = pd.read_csv(DATA / "test.csv", dtype={"shipping": "int8"}, parse_dates=["listed_at"])
print("train:", train.shape, " test:", test.shape)

# STEP 2: 型一覧。ここに最初のヒントが隠れている
print("\n--- train.dtypes ---")
print(train.dtypes)
print("\n注目1: listed_at が datetime64 → parse_dates が効いている")
print("注目2: description_len が float64 → 説明文の文字数は整数のはず。float = 欠損(NaN)がある証拠")
print("注目3: 文字列列の dtype が 'object' ではなく 'str' → これが pandas 3.0")

# STEP 3: 先頭を見る
print("\n--- train.head(3) ---")
print(train.head(3))

# STEP 4: info() — 列ごとの non-null 件数。欠損のある列がひと目で分かる
print("\n--- train.info() ---")
train.info()

# STEP 5: describe() — 数値列の要約
print("\n--- train.describe() ---")
print(train.describe().T)
print("\nprice の平均 18184 に対して中央値 7019、最大 3580200 → 桁違いの値が混ざっている疑い")
print("views の最小値が -1 → 閲覧数が負になることはあり得ない = 収集側のバグ")

# STEP 6: Series と DataFrame の違いを型と shape で確認する
one = train["price"]        # 角括弧1つ
two = train[["price"]]      # 角括弧2つ
print("\ntrain['price']    ->", type(one).__name__, "shape:", one.shape)
print("train[['price']]  ->", type(two).__name__, "shape:", two.shape)
print("train[['category','price']] ->", type(train[["category", "price"]]).__name__,
      "shape:", train[["category", "price"]].shape)

# STEP 7: pandas 3.0 の dtype 差分を実物で確認
print("\nselect_dtypes(include='str')    :", list(train.select_dtypes(include="str").columns))
print("select_dtypes(include='object') :", list(train.select_dtypes(include="object").columns), "← 1.x 系の書き方だと空振り")

## ④ 予測: フィルタの結果と、代入の落とし穴

train は **3040行 × 9列**。`price` が 0円 の行は **21件**ある(③ の `describe()` で最小値0を見たはず)。

次のセルを実行する前に予測しよう。

1. `train[train["price"] > 0].shape` は?
2. `train.loc[train["category"] == "家電", ["category", "price"]]` の **列数**は? 返る型は Series と DataFrame のどちら?
3. `train[train["price"] > 0] & ...` のような複数条件を書くとき、`and` を使うとどうなる?
4. **`demo[demo["price"] > 5000]["price"] = 0`** を実行したあと、`demo` の中身は書き換わっている? いない?

In [ ]:
# GOAL: NumPy のブールマスクがそのまま効くことと、pandas 3.0 での代入の作法を確認する

# STEP 1: 単一条件のフィルタ。行数の変化を必ず print する
mask_positive = train["price"] > 0          # ← ブール値の Series(NumPy のブール配列 + 行ラベル)
print("mask の型:", type(mask_positive).__name__, "/ dtype:", mask_positive.dtype, "/ True の件数:", int(mask_positive.sum()))
print("フィルタ前:", train.shape, "→ フィルタ後:", train[mask_positive].shape)

# STEP 2: 複数条件は & | ~ と括弧。既習の NumPy とまったく同じ規則
both = train[(train["price"] > 0) & (train["shipping"] == 1)]
print("\nprice>0 かつ shipping==1 :", both.shape)
either = train[(train["price"] == 0) | (train["views"] < 0)]
print("price==0 または views<0   :", either.shape)

# STEP 3: and を使うとどうなるか(例外を捕まえて中身を見る)
try:
    _ = train[(train["price"] > 0) and (train["shipping"] == 1)]
except ValueError as e:
    print("\nand を使うと ValueError:", str(e)[:90], "...")
    print("→ 『配列全体が真か偽か』を1つの真偽値に潰そうとして失敗している(NumPy と同じ現象)")

# STEP 4: .loc で行と列を同時に切る
kaden = train.loc[train["category"] == "家電", ["category", "price"]]
print("\n.loc[行条件, 列リスト] ->", type(kaden).__name__, kaden.shape)
kaden_price = train.loc[train["category"] == "家電", "price"]   # 列を文字列1つで指定すると Series
print(".loc[行条件, '列名']   ->", type(kaden_price).__name__, kaden_price.shape)

# STEP 5: pandas 3.0 の chained assignment — 代入が「効かない」ことを見る
demo = train[["item_id", "price"]].head(6).copy()
print("\n--- 代入前 ---")
print(demo["price"].tolist())

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    demo[demo["price"] > 5000]["price"] = 0          # ← 効かない書き方
print("出た警告:", [type(w.message).__name__ for w in caught])
print("代入後(効いていない):", demo["price"].tolist())

demo.loc[demo["price"] > 5000, "price"] = 0          # ← 正しい書き方
print("\n.loc で代入した後:", demo["price"].tolist())

## ⑥ 書いてみる: 条件で絞って、列を選ぶ

「**送料込み(`shipping == 1`)で、かつ価格が正常(`price > 0`)な出品だけ**を、
`category` と `price` の2列で取り出す」— コンペで毎日やる操作だ。

次のセルで `sub_b` を作ろう。

- 行の条件: `shipping` が `1` **かつ** `price` が `0` より大きい
- 列: `category` と `price` の**2列だけ**、**この順**で
- 結果は **DataFrame**(Series ではない)
- 作れたら `print(sub_b.shape)` して行数を確認する癖をつける

ヒント: 行条件と列を同時に指定できる書き方が ⑤ の STEP 4 にある。条件は必ず**それぞれ括弧で囲む**。

In [ ]:
sub_b = None
# ここに書く(ヒント: df.loc[行条件, ["列1", "列2"]]。条件を & でつなぐときは (…) & (…) と括弧で囲む)

print("sub_b:", None if sub_b is None else sub_b.shape)

In [ ]:
# ===== チェックポイント 2: 行フィルタと列選択 =====
check_frame("B-1 形と列名", sub_b, shape=(1765, 2), columns=["category", "price"],
            hint="行数が 3040 のままなら条件が効いていない。行数が 1775 なら price>0 を忘れている。"
                 "列が3つ以上なら列リストを指定していない。列順は ['category', 'price']。")

check("B-2 price の最小値(0円が除かれているか)",
      frame_stat(sub_b, "price", "min"), 522.0,
      hint="price > 0 の条件が入っていれば 0 は残らない。")

check("B-3 price の平均",
      frame_stat(sub_b, "price", "mean"), 18573.931444759208,
      hint="桁違いの外れ値がまだ入っているので平均は大きい。これは想定どおり(概念3で扱う)。")

check("B-4 sub_b は DataFrame か(Series ではない)",
      1.0 if isinstance(sub_b, pd.DataFrame) else 0.0, 1.0,
      hint="列を文字列1つで指定すると Series になる。['category', 'price'] とリストで渡す。")

print("\n(4つとも [OK] になったら次の概念へ)")

---
# 概念3 — データ健全性チェック: モデルの前に入力を疑う

## ① なぜ: 「精度が出ない」の原因の多くはモデルではなく入力

実務でモデルの精度が出ないとき、原因の内訳はだいたいこうなる。

- 上流の収集スクリプトのバグ(閲覧数が `-1`、金額が `null` ではなく `0` で入る)
- 入力フォームの事故(桁を間違えて100倍で登録された商品)
- **同じレコードの二重収集**(同じ商品を2回クロールした)
- 学習時と本番でカテゴリ体系が違う(**本番にだけ現れる新カテゴリ**)
- 欠損が**特定の群に偏っている**(「ブランド名が空」が特定カテゴリに集中している)

君の会社のように Web からデータを集めている場合、これらは例外ではなく**日常**だ。
そして重要なのは、**壊れた入力の上でモデルを比較しても議論が成立しない**こと。
「LightGBM と線形回帰、どちらが良いか」の前に「そのデータ、そもそも正しいか」を通す。
だから健全性チェックは、必ず**モデルより先**に置く。

このユニットのデータには、上のすべてが**意図的に仕込んである**。全部見つけよう。

## ② 解説: 5つのチェックと、そのための API

### 必ず通す5つのチェック

| # | チェック | 見つかるもの |
|---|---|---|
| 1 | **欠損**: 列ごとの件数と率、そして**群ごとの率** | 「全体では11%だが、ある群では48%」のような偏り |
| 2 | **目的変数の異常値**: 0・負・桁違い | 誤入力、`null` の代わりの `0` |
| 3 | **カテゴリの分布**: 値の内訳 | 想定外の値、極端に少ない水準 |
| 4 | **重複行** | 二重収集(unit02 で「リークの原因」として再登場する) |
| 5 | **train と test の突き合わせ**: 列・dtype・カテゴリ集合 | **test にしかないカテゴリ**、型のズレ |

### API 一覧

| 用途 | API | 戻り値 | 注意 |
|---|---|---|---|
| 欠損かどうか | `df.isna()` | 同じ形のブール DataFrame | `isnull()` は同じものの別名 |
| 列ごとの欠損**件数** | `df.isna().sum()` | Series(列名 → 件数) | ブールの `sum()` は True の個数 |
| 列ごとの欠損**率** | `df.isna().mean()` | Series(列名 → 割合) | True=1 の平均 = 割合。C# の `list.Count(x => x.Brand == null) / (double)list.Count` |
| 値の内訳 | `s.value_counts()` | Series(値 → 件数) | **既定で欠損を除く**。含めるなら `dropna=False` |
| 割合で内訳 | `s.value_counts(normalize=True)` | Series(値 → 割合) | 合計1 |
| ユニーク値 | `s.unique()` / `s.nunique()` | 配列 / int | |
| 重複行の判定 | `df.duplicated(subset=[...])` | ブール Series | **既定は2件目以降が True**(1件目は残る) |
| 重複削除 | `df.drop_duplicates(subset=[...])` | DataFrame | **削除の前後で必ず shape を print** |
| 欠損を埋める | `s.fillna(値)` | Series | 元は変えず、新しい Series を返す |
| 欠損行を落とす | `df.dropna(subset=[...])` | DataFrame | 落とし過ぎに注意。まず率を見る |

### `subset` に何を渡すかが本質

「重複行」とは何か? `item_id` は行ごとにユニークに振られているので、**`item_id` を含めて比較したら重複はゼロ**になる。
知りたいのは「**中身が同じ出品が2回入っていないか**」だから、`item_id` を**除いた**全列で比較する必要がある。

```python
feature_cols = [c for c in df.columns if c != "item_id"]   # ← 内包表記
```

これが Python の**リスト内包表記**。TypeScript の
`const cols = df.columns.filter(c => c !== "item_id")` とまったく同じことを書いている。
`[式 for 変数 in 反復可能 if 条件]` の順で読む。

In [ ]:
# GOAL: 欠損・異常値・分布・重複を、それぞれ1〜2行の API で洗い出す

# STEP 1: 欠損の件数と率を1枚にまとめる
#   pd.DataFrame({...}) は「列名 -> Series」の対応表から表を作る(C# の new { A = ..., B = ... } に近い)
na_report = pd.DataFrame({
    "欠損件数": train.isna().sum(),
    "欠損率": train.isna().mean().round(4),
})
print("--- train の欠損 ---")
print(na_report[na_report["欠損件数"] > 0])
print("\n→ brand が約11%、description_len が約4%。description_len が float64 だった理由がこれ。")

# STEP 2: 目的変数の異常値。ブール Series の .sum() が「該当件数」
print("\n--- price の異常値 ---")
print("price == 0        :", int((train["price"] == 0).sum()), "件  ← 0円の商品は存在しないはず")
print("price > 1,000,000 :", int((train["price"] > 1_000_000).sum()), "件  ← 桁違いの誤入力(100倍)の疑い")
print("price が大きい順の上位5件:", sorted(train["price"], reverse=True)[:5])

# STEP 3: 数値列の異常値。views は閲覧数なので負にならないはず
print("\n--- views の異常値 ---")
print("views < 0 :", int((train["views"] < 0).sum()), "件")
print("負値の中身:", train.loc[train["views"] < 0, "views"].unique(), "← -1 は『取得失敗』のセンチネル値と推測できる")

# STEP 4: カテゴリの内訳(件数と割合)
print("\n--- category の内訳(割合)---")
print(train["category"].value_counts(normalize=True).round(3))
print("\n--- condition の内訳(件数)---")
print(train["condition"].value_counts())
print("\n--- brand の内訳(欠損も含める)---")
print(train["brand"].value_counts(dropna=False).head(4))

# STEP 5: 重複行。item_id を除いた全列で判定する
feature_cols = [c for c in train.columns if c != "item_id"]
print("\n--- 重複行 ---")
print("item_id を含めて判定:", int(train.duplicated().sum()), "件  ← IDがユニークなので当然0")
print("item_id を除いて判定:", int(train.duplicated(subset=feature_cols).sum()), "件  ← これが本当の重複")

deduped = train.drop_duplicates(subset=feature_cols)
print("削除前:", train.shape, "→ 削除後:", deduped.shape, " 差分:", train.shape[0] - deduped.shape[0], "行")

## ④ 予測: 全体の平均は嘘をつく

③ で `brand` の欠損率は **全体で約 11%** と出た。「1割くらい欠けているのか、まあ埋めればいいか」で済ませたくなる。

だが `brand`(ブランド名)という列の意味を考えてほしい。**家電にはブランドがあるが、本や CD に「ブランド」はあまり無い**。
つまりこの欠損は**ランダムにばらまかれていない**可能性が高い。

次のセルを実行する前に予測しよう。

1. `brand` の欠損率を**カテゴリ別**に出したら、最も高いカテゴリはどれ? だいたい何%くらいだと思う?
2. 最も低いカテゴリとの差はどのくらいつく?
3. **`test` にあって `train` に無いカテゴリ値**は存在すると思う? もし存在したら、
   「カテゴリ別に価格の中央値を出して予測する」という作戦は何が起きる?
4. `train` と `test` で**列の dtype** はすべて一致していると思う?

> なぜこれが致命的か: 欠損が特定の群に偏っているなら、「全体の最頻ブランドで埋める」という素朴な処理は
> **本・音楽カテゴリの行に、実在しないブランド情報を大量に捏造する**ことになる。

In [ ]:
# GOAL: 「全体の率」を群で割り、train と test を突き合わせて、隠れた偏りとズレを見つける

# STEP 1: brand の欠損率をカテゴリ別に出す
#   for 文で回して、マスクで絞ってから isna().mean() を取る。
#   f"..." は f-string(C# の $"..." と同じ文字列補間)。
#   {rate:6.1%} は「幅6・小数1桁のパーセント表示」の書式指定。
print("--- brand の欠損率(全体 {:.1%})---".format(train["brand"].isna().mean()))
for cat in sorted(train["category"].unique()):
    mask = train["category"] == cat
    rate = train.loc[mask, "brand"].isna().mean()
    bar = "#" * int(rate * 40)
    print(f"  {cat:<8} 件数={int(mask.sum()):>5}  欠損率={rate:6.1%}  {bar}")

print("\n→ 全体 11% は『どの行も1割くらい欠けている』という意味ではなかった。")
print("  本・音楽では約半分が欠損。ここを一律に埋めると、存在しないブランドを捏造することになる。")

# STEP 2: train と test の列を突き合わせる
print("\n--- 列の突き合わせ ---")
print("train にしかない列:", sorted(set(train.columns) - set(test.columns)))
print("test にしかない列 :", sorted(set(test.columns) - set(train.columns)))

# STEP 3: dtype を突き合わせる(共通列だけ)
common = [c for c in train.columns if c in test.columns]
dtype_diff = pd.DataFrame({"train": train[common].dtypes, "test": test[common].dtypes})
dtype_diff["一致"] = dtype_diff["train"] == dtype_diff["test"]
print("\n--- dtype の突き合わせ ---")
print(dtype_diff)

# STEP 4: カテゴリ集合を突き合わせる。ここが今日いちばん重要なチェック
cats_train = set(train["category"].dropna())
cats_test = set(test["category"].dropna())
print("\n--- category 集合の突き合わせ ---")
print("train のカテゴリ  :", sorted(cats_train))
print("test のカテゴリ   :", sorted(cats_test))
print("test にしかない値 :", sorted(cats_test - cats_train), "  ← 未知カテゴリ!")
unseen = ~test["category"].isin(cats_train)   # ~ はブール Series の否定(C# の ! に相当)
#   .isin(集合) は「その値が集合に含まれるか」のブール Series。~ で反転させて「含まれない行」を取る。
print("未知カテゴリの行数:", int(unseen.sum()), "/", len(test))
print("\n→ 『カテゴリ別の中央値を貼る』作戦だと、この", int(unseen.sum()), "行だけ予測値が作れず欠損になる。")
print("  提出ファイルに欠損があると、その提出は丸ごと弾かれる。必ず退避先(全体の中央値など)を用意する。")

## ⑥ 書いてみる: 健全性チェックを1つの辞書にまとめる

調査の結果は、あとで再実行して比較できるように**構造化して残す**のが実務の作法だ
(「先週と比べて欠損率が急に上がっていないか」を毎日回す = unit10 の本番監視につながる)。

ここでは Python の **dict**(辞書)にまとめる。
TypeScript の `Record<string, number>` とほぼ同じもので、書き方はこう:

```python
result = {"キー1": 値1, "キー2": 値2}
result["キー1"]        # 取り出し(TS の result["キー1"] と同じ)
```

次のセルで `sanity` という dict を作り、**次の4つのキー**に**整数**を入れよう。集計対象は `train`。

| キー | 意味 |
|---|---|
| `"price_zero"` | `price` が **ちょうど 0** の行数 |
| `"price_huge"` | `price` が **1,000,000 より大きい** 行数(`>=` ではなく `>`) |
| `"views_negative"` | `views` が **0 未満** の行数 |
| `"dup_rows"` | **`item_id` 以外の全列が一致する重複行**の数(2件目以降を数える) |

ヒント: ブール Series の `.sum()` が「True の件数」。重複は ③ の STEP 5 と同じやり方。
値は `int(...)` で包んで整数にしておくと後で扱いやすい。

In [ ]:
sanity = {}
# ここに書く(ヒント: (train["price"] == 0).sum() のようなブール Series の合計。
#           重複は item_id を除いた列リストを subset に渡す)

print(sanity)

In [ ]:
# ===== チェックポイント 3: 健全性チェック =====
_g = sanity.get if isinstance(sanity, dict) else (lambda k: None)

check("C-1 price == 0 の行数", _g("price_zero"), 21,
      hint="(train['price'] == 0).sum() — 等価比較は == を2つ。")

check("C-2 price > 1,000,000 の行数", _g("price_huge"), 11,
      hint="閾値は 1_000_000(Python では数値に _ を入れて桁区切りにできる)。>= ではなく > で数える。")

check("C-3 views < 0 の行数", _g("views_negative"), 40,
      hint="収集失敗を表す -1 が入っている行。")

check("C-4 item_id を除いた重複行の数", _g("dup_rows"), 40,
      hint="0 になったなら subset を渡し忘れている(item_id が全行ユニークなので重複ゼロになる)。"
           "80 になったなら keep=False を使っている(既定の『2件目以降』で数える)。")

check("C-5 キーが4つそろっているか",
      len(sanity) if isinstance(sanity, dict) else None, 4,
      hint="キー名は price_zero / price_huge / views_negative / dup_rows の4つちょうど。")

print("\n(5つとも [OK] になったら次の概念へ)")

---
# 概念4 — 最初の提出

## ① なぜ: 初日に submission.csv を出す

Kaggle の上位陣がほぼ全員やっているのが「**初日に、雑でいいから1本提出する**」ことだ。理由は3つ。

1. **提出パイプラインが通ることを最初に確認する。** 締切1時間前に「列名が違う」で弾かれるのが最悪のシナリオ。
   フォーマット検証は最初に作って、以後ずっと使い回す。
2. **ベースラインが無いと改善したか分からない。** 「CV が 0.53 でした」は、比較対象がなければ良いのか悪いのか判断できない。
   定数提出のスコアが「何もしなかったときの点」であり、これを下回るモデルは**存在価値がない**。
3. **データ理解が提出という形で検証される。** 実際に提出物を作ろうとすると、
   「未知カテゴリの行はどうする?」のような未解決の穴が必ず表面化する(概念3 で見つけたやつだ)。

実務でも同じ。「最終的なモデル」より先に、**入口から出口まで通る一番細いパイプ**を作る。
C# で言えば、機能を作り込む前にまず I/O の契約とスモークテストを通しておくのと同じ発想だ。

## ② 解説: 定数 → 群別 → モデル、の順に上げていく

### ベースラインの階段

| 段 | 予測 | このユニットで作る | 必要な道具 |
|---|---|---|---|
| 0 | `sample_submission` そのまま | 出す | なし |
| 1 | **全行に同じ定数**(中央値) | 出す | 中央値 |
| 2 | **カテゴリ別の中央値** | 出す | 群ごとの統計 + 未知カテゴリの退避 |
| 3 | 数値列を使った単純モデル | 演習 ex03 で | 検証データの分割 |
| 4 | GBDT | unit03 で | LightGBM |

**RMSLE を使うので、平均ではなく中央値を出す**(概念1 の ⑤ で実測した通り)。
そしてその中央値は、**掃除したあとのデータ**から計算しなければならない。
0円の行と100倍の誤入力が混ざったままだと、中央値も平均も歪む。**概念3 の結果がここで効く**。

### 提出フォーマットの検証を関数化する

毎回目視で確認するのは事故のもと。**5項目の検査を関数にして、提出直前に必ず通す**。

| # | 検査 | 落ちる典型例 |
|---|---|---|
| 1 | 行数が test と一致 | フィルタしたまま提出した |
| 2 | ID 列の集合が test と一致 | 並べ替えや欠損で ID が欠けた |
| 3 | 列名が `sample_submission` と一致(順序も) | 大文字小文字違い、余計な列 |
| 4 | 予測列に**欠損がゼロ** | 未知カテゴリで `NaN` が出た |
| 5 | 予測列に**負値がない**(価格なので) | モデルが負を吐いた |

C# で言えば「**出力の契約に対するユニットテスト**」。書いておけば以後ずっと守ってくれる。

### 対応表を Series に貼る: `Series.map(dict)`

「カテゴリ名 → 中央値」の対応表(dict)を作り、`test` の各行のカテゴリをその表で引く。

```python
table = {"家電": 19955.5, "本・音楽": 1797.0}
test["category"].map(table)     # 各行のカテゴリを table で引いた Series が返る
```

TypeScript の `arr.map(x => table[x])` とほぼ同じ。ただし決定的な違いがひとつ:
**表に無いキーは `NaN`(欠損)になる**。ここが概念3 で見つけた未知カテゴリと直結する。

### `to_csv(index=False)` の `index=False`

pandas の DataFrame は**行ラベル(インデックス)**を持っている。`to_csv` は既定でそれも書き出すので、
`index=False` を忘れると**先頭に名前のない列が1本増えた CSV** ができる。
提出システムは列数の違いで即エラーを返す。**Kaggle 初心者が最も多く踏む地雷**なので、⑤ で実物を見る。

In [ ]:
# GOAL: 掃除 → 定数提出 → フォーマット検証 → 保存 を一直線に通す(各段階で shape を print)

# STEP 1: 学習データを掃除する。1操作ごとに行数がどう変わったかを必ず出す。
print("--- クリーニング ---")
feature_cols = [c for c in train.columns if c != "item_id"]

clean = train.drop_duplicates(subset=feature_cols)
print(f"  重複行を削除     : {train.shape} -> {clean.shape}  ({train.shape[0] - clean.shape[0]} 行減)")

before = clean.shape[0]
clean = clean[clean["price"] > 0]
print(f"  price==0 を除外   : ({before}, 9) -> {clean.shape}  ({before - clean.shape[0]} 行減)")

before = clean.shape[0]
clean = clean[clean["price"] <= 1_000_000]
print(f"  桁違いを除外      : ({before}, 9) -> {clean.shape}  ({before - clean.shape[0]} 行減)")
print(f"  合計 {train.shape[0]} -> {clean.shape[0]} 行({(1 - clean.shape[0] / train.shape[0]):.1%} を除去)")

# STEP 2: 掃除の前後で代表値がどう動いたか
print("\n--- 掃除の効果 ---")
print(f"  掃除前 中央値={train['price'].median():>10.1f}  平均={train['price'].mean():>10.1f}")
print(f"  掃除後 中央値={clean['price'].median():>10.1f}  平均={clean['price'].mean():>10.1f}")
print("  → 中央値はほとんど動かない(頑健)。平均は大きく動く。これも中央値を選ぶ理由。")

MEDIAN = float(clean["price"].median())
print("\nMEDIAN =", MEDIAN)

# STEP 3: sample_submission をひな型にして定数提出を作る。ゼロから組み立てない。
sub_const = sample_submission.copy()
sub_const["price"] = MEDIAN
print("\n--- 定数提出 ---")
print("shape:", sub_const.shape, " 列:", list(sub_const.columns))
print(sub_const.head(3))


# STEP 4: 提出フォーマットの検証を関数化する。問題点のリストを返す(空リスト = 合格)。
def validate_submission(sub, expected_ids, id_col="item_id", pred_col="price"):
    """提出ファイルとして成立しているかを検査し、問題点の文字列リストを返す。空なら合格。"""
    if not isinstance(sub, pd.DataFrame):
        return [f"DataFrame ではない(実際: {type(sub).__name__})"]
    problems = []
    if list(sub.columns) != [id_col, pred_col]:
        problems.append(f"列名が違う: 期待 {[id_col, pred_col]} / 実際 {list(sub.columns)}")
    if len(sub) != len(expected_ids):
        problems.append(f"行数が違う: 期待 {len(expected_ids)} / 実際 {len(sub)}")
    if id_col in sub.columns and set(sub[id_col]) != set(expected_ids):
        problems.append("item_id の集合が test と一致しない")
    if pred_col in sub.columns:
        s = sub[pred_col]
        if s.isna().any():
            problems.append(f"{pred_col} に欠損が {int(s.isna().sum())} 件ある")
        if not pd.api.types.is_numeric_dtype(s):
            problems.append(f"{pred_col} が数値型でない: {s.dtype}")
        elif (s < 0).any():
            problems.append(f"{pred_col} に負値が {int((s < 0).sum())} 件ある")
    return problems


def report(name, sub, expected_ids):
    problems = validate_submission(sub, expected_ids)
    if problems:
        print(f"[提出NG] {name}")
        for p in problems:
            print("   -", p)
    else:
        print(f"[提出OK] {name}  行数={len(sub)}")
    return problems


print("\n--- 検証 ---")
report("sub_const", sub_const, test["item_id"])

# STEP 5: 保存。index=False を忘れない。
path_const = OUT / "submission_constant.csv"
sub_const.to_csv(path_const, index=False)
print("\n保存:", path_const)
print("読み直して確認:", pd.read_csv(path_const).shape, list(pd.read_csv(path_const).columns))

## ④ 予測: 2つの地雷を踏む前に

次のセルでは、わざと2つの間違いを踏んでみる。実行前に予測しよう。

**地雷1: `index=False` を忘れる**

`sub_const.to_csv(path)` と書いて(`index=False` なし)保存し、`pd.read_csv` で読み直す。

1. 読み直した DataFrame の `shape` は? (元は `(1000, 2)`)
2. 列名はどうなる?

**地雷2: 対応表に無いカテゴリを `map` で引く**

`test["category"]` を、**「家電」と「本・音楽」の2つしか入っていない**対応表 `demo_median` で引く。
test は 1000行で、内訳は 家電 232件 / 本・音楽 137件 / その他 631件。

3. `test["category"].map(demo_median)` の**欠損は何件**になる?
4. その結果をそのまま `price` 列にして `validate_submission` に通すと、何番目の検査で落ちる?
5. 概念3 で見つけた **`ベビー・キッズ`(test にしか無いカテゴリ)** は、対応表を全カテゴリ分そろえても救えるか?

In [ ]:
# GOAL: 提出でよく踏む2つの地雷を、実際に踏んで挙動を見る

# ===== 地雷1: index=False を忘れる =====
bad_path = OUT / "_broken_index.csv"
sub_const.to_csv(bad_path)                 # ← index=False を書き忘れた
reloaded = pd.read_csv(bad_path)
print("--- 地雷1: index=False 忘れ ---")
print("保存前:", sub_const.shape, list(sub_const.columns))
print("読み直し:", reloaded.shape, list(reloaded.columns), "  ← 名前のない列が増えている")
print(reloaded.head(2))
print("\n検証にかけると:")
report("index=False を忘れた提出", reloaded, test["item_id"])
bad_path.unlink()   # 後片付け

# ===== 地雷2: 対応表に無いカテゴリ =====
print("\n--- 地雷2: map と未知のキー ---")
demo_median = {"家電": 19955.5, "本・音楽": 1797.0}    # わざと2カテゴリだけの対応表
mapped = test["category"].map(demo_median)
print("map の結果の型:", type(mapped).__name__, " shape:", mapped.shape)
print("欠損の件数:", int(mapped.isna().sum()), "/", len(test))
print("欠損になったカテゴリの内訳:")
print(test.loc[mapped.isna(), "category"].value_counts())

sub_broken = pd.DataFrame({"item_id": test["item_id"], "price": mapped})
print("\n検証にかけると:")
report("対応表が足りない提出", sub_broken, test["item_id"])

# STEP 3: 退避先を用意すれば直る。fillna(値) は欠損だけをその値で埋めた新しい Series を返す。
fixed = mapped.fillna(MEDIAN)
sub_fixed = pd.DataFrame({"item_id": test["item_id"], "price": fixed})
print("\nfillna(MEDIAN) で埋めたあと:")
report("退避先を入れた提出", sub_fixed, test["item_id"])
print("\n→ 未知カテゴリは『避けられない前提』として、必ず退避先を設計に入れておく。")

## ⑥ 書いてみる: カテゴリ別中央値で提出を作る

いよいよ2本目の提出だ。定数提出より一段賢い「**カテゴリごとの中央値**」を出す。

次のセルで、以下の2つを作ろう。

**1. `cat_median`** — `clean`(D③ で作った掃除済みデータ)から、
   **カテゴリ名 → そのカテゴリの `price` 中央値** の dict を作る。キーは `clean` に登場する5カテゴリ。

  - 群ごとの値を出すには、概念3 の ⑤ でやった「マスクで絞る」やり方がそのまま使える:
    `clean.loc[clean["category"] == c, "price"].median()`
  - dict は for 文で1つずつ入れてもいいし、**辞書内包表記** `{k: v for k in ...}` でもよい
    (TypeScript の `Object.fromEntries(cats.map(c => [c, f(c)]))` と同じ発想)
  - 値は `float(...)` で包んでおく

**2. `sub_cat`** — `sample_submission` と同じ形の DataFrame。

  - 列は `item_id`, `price` の2列、**この順**
  - `item_id` は `test["item_id"]` をそのまま使う
  - `price` は `test["category"]` を `cat_median` で引いた値。
    **`ベビー・キッズ` は `cat_median` に無い**ので、`MEDIAN`(全体の中央値 = 7052.0)で埋める

作れたら、`report("sub_cat", sub_cat, test["item_id"])` を呼んで検証を通し、
`OUT / "submission_category_median.csv"` に `index=False` で保存しよう。5〜8行で書ける。

In [ ]:
cat_median = {}
sub_cat = None
# ここに書く(ヒント: clean.loc[clean["category"] == c, "price"].median() を全カテゴリぶん集めて dict にする。
#           貼るのは test["category"].map(cat_median)、未知カテゴリは .fillna(MEDIAN))

print("cat_median:", cat_median)
if isinstance(sub_cat, pd.DataFrame):
    print("sub_cat:", sub_cat.shape)
    report("sub_cat", sub_cat, test["item_id"])
    sub_cat.to_csv(OUT / "submission_category_median.csv", index=False)
    print("保存:", OUT / "submission_category_median.csv")

In [ ]:
# ===== チェックポイント 4: カテゴリ別中央値の提出 =====
check("D-1 cat_median のキー数",
      len(cat_median) if isinstance(cat_median, dict) else None, 5,
      hint="clean に登場する category は5種類(ベビー・キッズ は test にしか無いので入らない)。")

check("D-2 cat_median['家電']",
      cat_median.get("家電") if isinstance(cat_median, dict) else None, 19955.5,
      hint="train ではなく clean(重複削除・0円除外・桁違い除外の済んだ表)から中央値を取る。")

check("D-3 cat_median['本・音楽']",
      cat_median.get("本・音楽") if isinstance(cat_median, dict) else None, 1797.0,
      hint="mean ではなく median。")

check_frame("D-4 sub_cat の形と列名", sub_cat, shape=(1000, 2), columns=["item_id", "price"],
            hint="行数は test と同じ 1000。列は ['item_id', 'price'] の順。"
                 "pd.DataFrame({'item_id': ..., 'price': ...}) で作れば順序はこの通りになる。")

check("D-5 price に欠損が無いか",
      frame_stat(sub_cat, "price", "isna_sum"), 0,
      hint="35件 欠損しているなら fillna(MEDIAN) を忘れている(ベビー・キッズ の行)。")

check("D-6 予測値の種類数",
      frame_stat(sub_cat, "price", "nunique"), 6,
      hint="5カテゴリの中央値 + 未知カテゴリ用の MEDIAN で、ちょうど6種類になる。")

check("D-7 予測値の平均",
      frame_stat(sub_cat, "price", "mean"), 8930.425,
      hint="ズレるなら、clean ではなく train から中央値を取っている可能性が高い。")

_baby = None
if isinstance(sub_cat, pd.DataFrame) and "price" in sub_cat.columns and len(sub_cat) == len(test):
    _m = (test["category"] == "ベビー・キッズ").to_numpy()
    if _m.any():
        _baby = float(sub_cat["price"].to_numpy(dtype=float)[_m][0])
check("D-8 未知カテゴリ(ベビー・キッズ)に入った値", _baby, 7052.0,
      hint="MEDIAN(掃除後の全体中央値)で埋める。")

print("\n(8つとも [OK] になったら答え合わせへ)")

## 答え合わせ: public LB と private LB

コンペ中は `test` の正解が見えない。だが今回は**教材なので、最後に答えを見せる**。

競技本番と同様、通常の学習環境では `test` の正解価格を配布しない。次のセルは、管理者用の正解ファイルを明示配置した場合だけ採点する。
概念1 で説明した通り、`test` を先頭30%(= public)と残り70%(= private)に分けて、それぞれ RMSLE を出す。

見るポイントは3つ:

1. **`sample_submission` そのまま → 定数提出 → カテゴリ別中央値** でスコアがどう動いたか
2. **public と private でスコアが一致しない**こと(同じ提出でも 0.0X くらい平気でズレる)。
   だから public LB の小さな上下に一喜一憂しても意味がない
3. モデルを1行も書かずに、**データ理解だけでスコアがここまで動く**こと

> ⑦ をまだ書いていない場合、②までのスコアだけが出る。書いてから戻ってきて再実行しよう。

In [ ]:
# GOAL: 自分の提出が実際に何点だったのかを見る(コンペ終了後の答え合わせという設定)

ANSWER = DATA / "_competition_answers_not_distributed.csv"

if not ANSWER.exists():
    print("答えファイルが見つかりません(このセルはスキップして構いません):", ANSWER)
else:
    answer = pd.read_csv(ANSWER)
    y_true = answer.set_index("item_id").loc[test["item_id"], "price"].to_numpy(dtype=float)
    n = len(y_true)
    k = int(n * 0.3)          # 先頭30% = public LB、残り70% = private LB という設定

    def _rmsle(a, b):
        a = np.asarray(a, dtype=float)
        b = np.asarray(b, dtype=float)
        return float(np.sqrt(np.mean((np.log1p(b) - np.log1p(a)) ** 2)))

    entries = [
        ("0. sample_submission そのまま", sample_submission["price"].to_numpy(dtype=float)),
        ("1. 定数提出(掃除後の中央値)", sub_const["price"].to_numpy(dtype=float)),
    ]
    _sc = globals().get("sub_cat")
    if isinstance(_sc, pd.DataFrame) and "price" in _sc.columns and len(_sc) == n and _sc["price"].notna().all():
        entries.append(("2. カテゴリ別中央値", _sc["price"].to_numpy(dtype=float)))
    else:
        print("(sub_cat が未完成なので 2 はスキップ。⑦ を書いてから再実行しよう)\n")

    print(f"{'提出':<34}{'public LB':>12}{'private LB':>12}")
    print("-" * 58)
    for label, p in entries:
        print(f"{label:<30}{_rmsle(y_true[:k], p[:k]):>12.5f}{_rmsle(y_true[k:], p[k:]):>12.5f}")
    print("\n※ RMSLE は小さいほど良い。")
    print("※ public(300件)と private(700件)で数値が違う。件数が少ないほど揺れる。")
    print("※ ここまでモデルは一行も書いていない。効いているのはデータの理解だけ。")

---
## 振り返り(自己評価 + TIL)

以下に**1〜2文ずつ**、自分の言葉で書いてみよう。書いた内容はセッション終了時の学習ノートと
スキルレベルの判定に使う(空欄でも先に進めるが、言語化すると定着が大きく変わる)。

**1. 今日学んだことを自分の言葉で:**

> (ここに書く)

**2. 難しかったこと・まだあやふやなこと:**

> (ここに書く)

**3. Feynman チェック — 次の3つに、資料を見ずに答えられる?**

- 「なぜこのコンペは RMSE ではなく RMSLE なのか」を、価格の分布に触れながら説明できる?
- 「`df[cond]["col"] = x` が効かない理由」を、Copy-on-Write に触れながら説明できる?
- 「`brand` の欠損率は全体で11%」という報告に対して、追加で何を確認すべきか言える?

> (ここに書く)

---
## まとめ

### 今日学んだこと

| # | 概念 | 一言でいうと |
|---|---|---|
| 1 | コンペの解剖 | train は答え付き、test は答えなし、`sample_submission` が**フォーマットの唯一の正解** |
| 2 | 評価指標を先に読む | 価格は裾が重い対数正規 → RMSE は高額品ゲームに化ける → **RMSLE**。指標が決まれば「平均より中央値」まで決まる |
| 3 | public / private LB | 最終順位は private。public の細かい上下を追うと shake で崩れる |
| 4 | pandas の型 | `df["col"]` は **Series**、`df[["col"]]` は **DataFrame**。角括弧の数で変わる |
| 5 | 行フィルタ | **既習の NumPy ブールマスクがそのまま効く**。`&` `\|` `~` と括弧も同じ規則 |
| 6 | `.loc[行, 列]` | 行と列を同時に切る。**代入するときは必ずこちら**(pandas 3.0 の Copy-on-Write) |
| 7 | shape を print する規律 | 読み込み・フィルタ・重複削除の各ステップで行数の変化を追う |
| 8 | 欠損の見方 | `isna().sum()` / `isna().mean()`、そして**群ごとに割る**。全体11%が特定群では48%だった |
| 9 | 異常値 | 0円・桁違い・負の閲覧数。`describe()` の min / max が最初の入口 |
| 10 | 重複行 | `duplicated(subset=item_id以外)`。ID を含めて判定すると必ず0件になる罠 |
| 11 | train / test の突き合わせ | 列・dtype・**カテゴリ集合**。test にしか無いカテゴリは必ず退避先を用意する |
| 12 | 提出の検証 | 行数・ID集合・列名・欠損ゼロ・負値なし の5項目を**関数化**して毎回通す |
| 13 | `to_csv(index=False)` | 忘れると名前のない列が増えて弾かれる |
| 14 | ベースラインの階段 | 定数 → 群別 → モデル。**定数提出を下回るモデルには存在価値がない** |

### この先どこで使うか(先読み)

- **unit02(検証設計とリーク)** — 今日見つけた**重複行**が、そのまま「同一実体リーク」の主役になる。
  train と valid に同じ商品が割れて入るとCVが跳ね上がる現象を、今日と同じ `duplicated` の発想で潰す。
  また今日書いた `rmsle` は、指標を自前実装して `sklearn.metrics` と突き合わせる回で再登場する。
- **unit03(GBDT)** — 今日の掃除済み `clean` に対して LightGBM を回す。
  「GBDT は欠損をそのまま食べられる」ので、今日見た `brand` の欠損をどう扱うかの判断が変わる。
- **unit04(特徴量エンジニアリング)** — 今日 for 文とマスクで書いた「カテゴリ別中央値」は、
  `groupby("category")["price"].transform("median")` の1行になる。
  そして**その統計を train 全体から作るとリークする**という話に進む(今日の作り方は実はリークしている!)。
- **unit05(テキスト)・unit08(画像)** — 「まず入力を疑う」「shape を print する」はそのまま持ち込む。
- **unit10(本番運用)** — 今日作った `sanity` dict は、本番の**データドリフト監視**の原型そのもの。
  毎日これを回して、欠損率やカテゴリ分布が前日から急変していないかを見る。
- **実務** — 新しいデータを受け取った日の最初の30分の型が、今日のレッスン1本分にあたる。

### 次にやること

**演習 `ex01_profile_and_sanity_check` へ進もう。lesson.ipynb を見ながらで OK。**
思い出せない API があれば、②の表に戻ればいい。暗記ではなく、**どこを見れば分かるか**を覚えているのが実務の状態だ。

演習は4本:

| 演習 | 内容 |
|---|---|
| `ex01_profile_and_sanity_check` | データ概要と健全性チェックを関数にまとめる |
| `ex02_align_and_clean` | train / test の突き合わせと掃除 |
| `ex03_holdout_score` | 手元で分割してスコアを測る(unit02 への橋渡し) |
| `ex04_capstone` | 読み込みから提出ファイル生成まで一気通貫 |